# RVC Next WebUI Colab
Colab NoteBook Created by [licyk](https://github.com/licyk)

Jupyter Notebook 仓库：[licyk/sd-webui-all-in-one](https://github.com/licyk/sd-webui-all-in-one)

这是适用于 [Colab](https://colab.research.google.com) 部署 [RVC Next WebUI](https://github.com/licyk/rvc-next-webui) 的 Jupyter Notebook，使用时请按顺序执行 Jupyter Notebook 单元。

Colab 链接：<a href="https://colab.research.google.com/github/licyk/sd-webui-all-in-one/blob/main/notebook/rvc_next_webui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 功能
1. 环境配置：配置安装的 PyTorch 版本、内网穿透的方式，并安装 [RVC Next WebUI](https://github.com/licyk/rvc-next-webui)，默认设置下已启用`配置环境完成后立即启动 RVC Next WebUI`选项，则安装完成后将直接启动 RVC Next WebUI，并显示访问地址。
2. 启动 RVC Next WebUI：启动 RVC Next WebUI，并显示访问地址。
3. 文件下载工具：使用链接下载文件（可选）。


## 使用
1. 在 Colab -> 代码执行程序 > 更改运行时类型 -> 硬件加速器 选择`GPU T4`或者其他 GPU。
2. `环境配置`单元中的选项通常不需要修改，保持默认即可。
3. 运行`环境配置`单元，默认设置下已启用`配置环境完成后立即启动 RVC Next WebUI`选项，则环境配置完成后立即启动 RVC Next WebUI，此时将弹出 Google Drive 授权页面，根据提示进行操作。配置完成后将启动 RVC Next WebUI，RVC Next WebUI 的访问地址和访问令牌可在日志中查看。
4. 如果未启用`配置环境完成后立即启动 RVC Next WebUI`选项，需要运行`启动 RVC Next WebUI`单元，此时将弹出 Google Drive 授权页面，根据提示进行操作。配置完成后将启动 RVC Next WebUI，RVC Next WebUI 的访问地址和访问令牌可在日志中查看。
5. 打开访问地址后，RVC Next WebUI 会要求输入访问令牌，填写日志中显示的访问令牌即可。

## 提示
1. Colab 在关机后将会清除所有数据，所以每次重新启动时必须运行`环境配置`单元才能运行`启动 RVC Next WebUI`单元。
2. [Ngrok](https://ngrok.com) 内网穿透在使用前需要填写 Ngrok Token，可在 [Ngrok](https://ngrok.com) 官网获取。
3. [Zrok](https://zrok.io) 内网穿透在使用前需要填写 Zrok Token, 可在 [Zrok](https://docs.zrok.io/docs/getting-started) 官网获取。
4. 每次启动 Colab 后必须运行`环境配置`单元，才能运行`启动 RVC Next WebUI`单元启动 RVC Next WebUI。
5. 内网穿透地址可被任何人访问，因此 RVC Next WebUI 会启用访问令牌。未填写`访问令牌`时将自动生成随机令牌，可在日志中查看；如需固定访问令牌，可在`环境配置`单元中填写。
6. RVC Next 所需的模型 (HuBERT、RMVPE 等) 在首次使用时由 RVC Next 自动下载。
7. 运行`启动 RVC Next WebUI`时将弹出 Google Drive 访问授权提示，根据提示进行授权。授权后，RVC Next WebUI 的`data`文件夹（设置、数据库、模型、实验和输出）将保存在 Google Drive 的`rvc_next_webui_output`文件夹中，重新启动 Colab 后仍可继续使用。
8. 在`额外挂载目录设置`中可以挂载一些额外目录，挂载后的目录可在 Google Drive 的`rvc_next_webui_output`文件夹中查看和修改。


In [ ]:
# @title 👇 环境配置
# SD WebUI All In One 功能初始化部分, 通常不需要修改
# 如果需要查看完整代码实现, 可阅读: https://github.com/licyk/sd-webui-all-in-one/blob/main/sd_webui_all_in_one
#################################################################################################################
import os
import sys
import shutil

try:
    _ = JUPYTER_ROOT_PATH  # type: ignore # noqa: F821
except Exception:
    JUPYTER_ROOT_PATH = os.getcwd()
try:
    _ = INIT_SD_WEBUI_ALL_IN_ONE_CORE  # type: ignore # noqa: F821
    print("[Core] SD WebUI All In One 核心模块已初始化")
except Exception:
    print("[Core] 初始化 SD WebUI All In One 核心模块中")
    !python -m pip install sd-webui-all-in-one --upgrade
    os.environ["SD_WEBUI_ALL_IN_ONE_IGNORE_INSTALL_DEP_TYPE"] = ""
    INIT_SD_WEBUI_ALL_IN_ONE_CORE = True
    from sd_webui_all_in_one import logger, VERSION, RvcNextWebUIManager

    logger.info("SD WebUI All In One 核心模块初始化完成, 版本: %s", VERSION)

#######################################################

# @markdown ## RVC Next WebUI 核心配置选项
# @markdown - RVC Next WebUI 安装路径：
RVC_NEXT_WEBUI_PATH = "/content/rvc-next-webui"  # @param {type:"string"}
# @markdown - RVC Next WebUI 启动参数：
RVC_NEXT_WEBUI_LAUNCH_ARGS = ""  # @param {type:"string"}
# @markdown - RVC Next WebUI 访问令牌（留空时自动生成随机令牌，访问 RVC Next WebUI 时需要填写该令牌）：
RVC_NEXT_WEBUI_ACCESS_TOKEN = ""  # @param {type:"string"}
# @markdown ---
# @markdown ## PyTorch 组件版本选项：
# @markdown - 安装 PyTorch 的类型：
PYTORCH_DEVICE_TYPE = "auto"  # @param ["auto", "cu118", "cu126", "cu128", "cu129", "cu130", "rocm6.2.4", "rocm6.3", "rocm6.4", "rocm7.1", "xpu", "cpu", "all"]
# @markdown - PyTorch（RVC Next 需要 2.7.1 及以上版本的 PyTorch）：
PYTORCH_VER = ""  # @param {type:"string"}
# @markdown - xFormers（自定义版本时需同时填写 PyTorch 版本）：
XFORMERS_VER = ""  # @param {type:"string"}
# @markdown ---
# @markdown ## 包管理器选项：
# @markdown - 使用 uv 作为 Python 包管理器
USE_UV = True  # @param {type:"boolean"}
# @markdown ---
# @markdown ## 镜像源选项：
# @markdown - 是否使用 PyPI 镜像
USE_PYPI_MIRROR = False  # @param {type:"boolean"}
# @markdown - PyPI 主镜像源
PIP_INDEX_MIRROR = "https://pypi.python.org/simple"  # @param {type:"string"}
# @markdown - PyPI 扩展镜像源
PIP_EXTRA_INDEX_MIRROR = "https://download.pytorch.org/whl/cu126"  # @param {type:"string"}
# @markdown - PyPI 额外镜像源
PIP_FIND_LINKS_MIRROR = ""  # @param {type:"string"}
# @markdown - 是否使用 Github 镜像
USE_GITHUB_MIRROR = False  # @param {type:"boolean"}
# @markdown - Github 镜像地址
CUSTOM_GITHUB_MIRROR = ""  # @param {type:"string"}
# @markdown - 是否使用 HuggingFace 镜像
USE_HF_MIRROR = False  # @param {type:"boolean"}
# @markdown - HuggingFace 镜像源地址
HUGGINGFACE_MIRROR = ""  # @param {type:"string"}
# @markdown ---
# @markdown ## 内网穿透选项：
# @markdown - 使用 remote.moe 内网穿透
USE_REMOTE_MOE = True  # @param {type:"boolean"}
# @markdown - 使用 localhost.run 内网穿透
USE_LOCALHOST_RUN = True  # @param {type:"boolean"}
# @markdown - 使用 pinggy.io 内网穿透
USE_PINGGY_IO = True  # @param {type:"boolean"}
# @markdown - 使用 CloudFlare 内网穿透
USE_CLOUDFLARE = True  # @param {type:"boolean"}
# @markdown - 使用 Gradio 内网穿透
USE_GRADIO = True  # @param {type:"boolean"}
# @markdown - 使用 Ngrok 内网穿透（需填写 Ngrok Token，可在 [Ngrok](https://ngrok.com) 官网获取）
USE_NGROK = True  # @param {type:"boolean"}
# @markdown - Ngrok Token
NGROK_TOKEN = ""  # @param {type:"string"}
# @markdown - 使用 Zrok 内网穿透（需填写 Zrok Token，可在 [Zrok](https://docs.zrok.io/docs/getting-started) 官网获取）
USE_ZROK = True  # @param {type:"boolean"}
# @markdown - Zrok Token
ZROK_TOKEN = ""  # @param {type:"string"}
# @markdown ---
# @markdown ## 快速启动选项：
# @markdown - 配置环境完成后立即启动 RVC Next WebUI（并挂载 Google Drive）
QUICK_LAUNCH = True  # @param {type:"boolean"}
# @markdown - 不重复配置环境（当重复运行环境配置时将不会再进行安装）
NO_REPEAT_CONFIGURE_ENV = True  # @param {type:"boolean"}
# @markdown ---
# @markdown ## 环境检查选项：
# @markdown - 多个任务名称用英文逗号分隔；同一任务同时包含和排除时，跳过优先。
# @markdown - 可用任务：`python-dependencies`、`torch-libomp`、`torch-version`
# @markdown - 仅执行的检查（留空表示全部）
ENV_INCLUDE_CHECKS = ""  # @param {type:"string"}
# @markdown - 跳过的检查（留空表示不跳过）
ENV_EXCLUDE_CHECKS = ""  # @param {type:"string"}
# @markdown ---
# @markdown ## 其他选项：
# @markdown - 清理无用日志
CLEAR_LOG = True  # @param {type:"boolean"}
# @markdown - 检查可用 GPU
CHECK_AVALIABLE_GPU = True  # @param {type:"boolean"}
# @markdown - 启用 TCMalloc 内存优化
ENABLE_TCMALLOC = True  # @param {type:"boolean"}
# @markdown - 启用 CUDA Malloc 显存优化
ENABLE_CUDA_MALLOC = True  # @param {type:"boolean"}
# @markdown - 更新内核
UPDATE_CORE = True  # @param {type:"boolean"}
# @markdown -  HuggingFace Token
HUGGINGFACE_TOKEN = ""  # @param {type:"string"}
# @markdown -  ModelScope Token
MODELSCOPE_TOKEN = ""  # @param {type:"string"}

# @markdown ---
##############################################################################
# @markdown ## 额外挂载目录设置, 在启动时挂载目录到 Google Drive：
EXTRA_LINK_DIR = []

# @markdown - cache 目录
link_cache_dir = False  # @param {type:"boolean"}
# @markdown - data 目录
link_data_dir = False  # @param {type:"boolean"}


link_cache_dir and EXTRA_LINK_DIR.append({"link_dir": "cache"})
link_data_dir and EXTRA_LINK_DIR.append({"link_dir": "data"})

#######################################################

if PYTORCH_DEVICE_TYPE == "auto":
    PYTORCH_DEVICE_TYPE = None

INSTALL_PARAMS = {
    "pytorch_mirror_type": PYTORCH_DEVICE_TYPE,
    "custom_pytorch_package": PYTORCH_VER or None,
    "custom_xformers_package": XFORMERS_VER or None,
    "use_pypi_mirror": USE_PYPI_MIRROR,
    "use_uv": USE_UV,
    "use_github_mirror": USE_GITHUB_MIRROR,
    "custom_github_mirror": CUSTOM_GITHUB_MIRROR or None,
    "use_hf_mirror": USE_HF_MIRROR,
    "pypi_index_mirror": PIP_INDEX_MIRROR or None,
    "pypi_extra_index_mirror": PIP_EXTRA_INDEX_MIRROR or None,
    "pypi_find_links_mirror": PIP_FIND_LINKS_MIRROR or None,
    "github_mirror": CUSTOM_GITHUB_MIRROR or None,
    "huggingface_mirror": HUGGINGFACE_MIRROR or None,
    "check_avaliable_gpu": CHECK_AVALIABLE_GPU,
    "enable_tcmalloc": ENABLE_TCMALLOC,
    "enable_cuda_malloc": ENABLE_CUDA_MALLOC,
    "custom_sys_pkg_cmd": None if sys.platform == "linux" and shutil.which("apt") else [],
    "huggingface_token": HUGGINGFACE_TOKEN or None,
    "modelscope_token": MODELSCOPE_TOKEN or None,
    "update_core": UPDATE_CORE,
}

INCLUDE_CHECKS = [name.strip() for name in ENV_INCLUDE_CHECKS.split(",") if name.strip()]
EXCLUDE_CHECKS = [name.strip() for name in ENV_EXCLUDE_CHECKS.split(",") if name.strip()]

CHECK_PARAMS = {
    "include_checks": INCLUDE_CHECKS or None,
    "exclude_checks": EXCLUDE_CHECKS or None,
    "use_uv": USE_UV,
    "use_pypi_mirror": USE_PYPI_MIRROR,
    "use_github_mirror": USE_GITHUB_MIRROR,
    "custom_github_mirror": CUSTOM_GITHUB_MIRROR or None,
}

TUNNEL_PARAMS = {
    "use_ngrok": USE_NGROK,
    "ngrok_token": NGROK_TOKEN or None,
    "use_cloudflare": USE_CLOUDFLARE,
    "use_remote_moe": USE_REMOTE_MOE,
    "use_localhost_run": USE_LOCALHOST_RUN,
    "use_gradio": USE_GRADIO,
    "use_pinggy_io": USE_PINGGY_IO,
    "use_zrok": USE_ZROK,
    "zrok_token": ZROK_TOKEN or None,
    "webui_name": "RVC Next WebUI",
}

try:
    _ = rvc_next_webui_manager_has_init  # type: ignore # noqa: F821
except Exception:
    rvc_next_webui = RvcNextWebUIManager(os.path.dirname(RVC_NEXT_WEBUI_PATH), os.path.basename(RVC_NEXT_WEBUI_PATH), port=7868)
    rvc_next_webui_manager_has_init = True

try:
    _ = rvc_next_webui_has_init  # type: ignore # noqa: F821
except Exception:
    rvc_next_webui_has_init = False

if NO_REPEAT_CONFIGURE_ENV:
    if not rvc_next_webui_has_init:
        rvc_next_webui.install(**INSTALL_PARAMS)
        INIT_CONFIG = 1
        rvc_next_webui_has_init = True
        CLEAR_LOG and rvc_next_webui.clear_output()
        logger.info("RVC Next WebUI 运行环境配置完成")
    else:
        logger.info("检测到不重复配置环境已启用, 并且在当前 Colab 实例中已配置 RVC Next WebUI 运行环境, 不再重复配置 RVC Next WebUI 运行环境")
        logger.info("如需在当前 Colab 实例中重新配置 RVC Next WebUI 运行环境, 请在快速启动选项中取消不重复配置环境功能")
else:
    rvc_next_webui.install(**INSTALL_PARAMS)
    INIT_CONFIG = 1
    CLEAR_LOG and rvc_next_webui.clear_output()
    logger.info("RVC Next WebUI 运行环境配置完成")

# 内网穿透转发的请求不是来自本机地址, RVC Next 需要监听所有地址并启用访问令牌
try:
    _ = RVC_NEXT_WEBUI_GENERATED_TOKEN  # type: ignore # noqa: F821
except Exception:
    import secrets

    RVC_NEXT_WEBUI_GENERATED_TOKEN = secrets.token_urlsafe(16)

RVC_NEXT_WEBUI_TOKEN = RVC_NEXT_WEBUI_ACCESS_TOKEN or RVC_NEXT_WEBUI_GENERATED_TOKEN
LAUNCH_CMD = rvc_next_webui.get_launch_command(["--host", "0.0.0.0", "--access-token", RVC_NEXT_WEBUI_TOKEN] + rvc_next_webui.parse_cmd_str_to_list(RVC_NEXT_WEBUI_LAUNCH_ARGS))

if QUICK_LAUNCH:
    logger.info("启动 RVC Next WebUI 中")
    os.chdir(RVC_NEXT_WEBUI_PATH)
    rvc_next_webui.check_env(**CHECK_PARAMS)
    rvc_next_webui.mount_drive(EXTRA_LINK_DIR)
    rvc_next_webui.apply_hf_mirror_env()
    with rvc_next_webui.tun_manager:
        rvc_next_webui.get_tunnel_url(**TUNNEL_PARAMS)
        logger.info("RVC Next WebUI 启动参数: %s", RVC_NEXT_WEBUI_LAUNCH_ARGS)
        logger.info("RVC Next WebUI 访问令牌: %s", RVC_NEXT_WEBUI_TOKEN)
        !{LAUNCH_CMD}
    CLEAR_LOG and rvc_next_webui.clear_output()
    logger.info("已关闭 RVC Next WebUI")

In [ ]:
#@title 👇 启动 RVC Next WebUI
try:
    _ = INIT_CONFIG
except Exception:
    raise Exception("未安装 RVC Next WebUI")

logger.info("启动 RVC Next WebUI")
os.chdir(RVC_NEXT_WEBUI_PATH)
rvc_next_webui.check_env(**CHECK_PARAMS)
rvc_next_webui.mount_drive(EXTRA_LINK_DIR)
rvc_next_webui.apply_hf_mirror_env()
with rvc_next_webui.tun_manager:
    rvc_next_webui.get_tunnel_url(**TUNNEL_PARAMS)
    logger.info("RVC Next WebUI 启动参数: %s", RVC_NEXT_WEBUI_LAUNCH_ARGS)
    logger.info("RVC Next WebUI 访问令牌: %s", RVC_NEXT_WEBUI_TOKEN)
    !{LAUNCH_CMD}
CLEAR_LOG and rvc_next_webui.clear_output()
logger.info("已关闭 RVC Next WebUI")

In [ ]:
#@title 👇 文件下载工具

#@markdown ### 填写文件的下载链接：
url = ""  #@param {type:"string"}
#@markdown ### 填写文件的保存路径：
file_path = "/content"  #@param {type:"string"}
#@markdown ### 填写文件的保存名称 (可选)：
filename = ""  #@param {type:"string"}

rvc_next_webui.download_file_from_url(
    url=url,
    path=file_path or None,
    save_name=filename or None,
)
